# QV-Select3 on matbench_steels

The code that produced `results.json.gz` (Matbench v0.1, official 5 folds, `MatbenchBenchmark.record`). It was run as a script: set `TASK = "matbench_steels"` and `OUT` to an output folder below, or run the same code from the command line as `python mb_run.py matbench_steels <out_dir>`. Model choice uses only each fold's training rows (inner 5-fold CV); the test fold is predicted once.


In [ ]:
import sys
sys.argv = ["mb_run.py", "matbench_steels", "out_steels"]


In [ ]:
"""Quantavane, materials field, first run: Matbench v0.1 composition tasks with the official recording.

Usage (from runs/fields/materials, MATMINER_DATA=$PWD/data):
  nice -n 10 .venv/bin/python -I code/mb_run.py <task> <out_dir>
    task: matbench_steels | matbench_expt_gap

Every model below is scored with MatbenchBenchmark.record on the official 5 folds and saved as its own
results.json.gz. Model choice inside "qv_select3" uses ONLY the training part of each fold (inner 5-fold CV):
the test fold is never seen before its single prediction.
"""
import json, os, sys, time, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
TASK, OUT = sys.argv[1], sys.argv[2]
os.makedirs(OUT, exist_ok=True)
NJ = 3  # heavily loaded machine: at most 3 threads per job
os.environ.setdefault("OMP_NUM_THREADS", str(NJ))

from matbench.bench import MatbenchBenchmark
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODEL_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "models")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# ---------- features (computed once for the whole dataset, unsupervised: no target used) ----------
mb0 = MatbenchBenchmark(autoload=False, subset=[TASK])
task0 = list(mb0.tasks)[0]
task0.load()
comps = task0.df["composition"].map(Composition)
elements = sorted({str(e) for c in comps for e in c.elements}, key=lambda s: Composition(s).elements[0].Z)
frac = pd.DataFrame([[c.get_atomic_fraction(e) for e in elements] for c in comps],
                    index=task0.df.index, columns=[f"x_{e}" for e in elements])
ep = ElementProperty.from_preset("magpie", impute_nan=True)
ep.set_n_jobs(1)
mag = pd.DataFrame(np.array([ep.featurize(c) for c in comps], dtype=float), index=task0.df.index,
                   columns=ep.feature_labels())
mag = mag.loc[:, mag.std() > 0]
FEATS = {"frac": frac, "magpie": mag, "frac+magpie": pd.concat([frac, mag], axis=1)}
log(TASK, "rows", len(frac), "elements", len(elements), "magpie cols", mag.shape[1])


def X_of(kind, comps_series):
    return FEATS[kind].loc[comps_series.index].values


# ---------- candidate models ----------
def tabpfn(seed):
    from tabpfn import TabPFNRegressor
    return TabPFNRegressor(device="cpu", n_estimators=int(os.environ.get("QV_TABPFN_N", 8)), random_state=seed,
                           model_path=os.path.join(MODEL_DIR, "tabpfn-v2-regressor.ckpt"),
                           ignore_pretraining_limits=True)


def gp(seed):
    return make_pipeline(StandardScaler(), GaussianProcessRegressor(
        ConstantKernel(1.0) * RBF(length_scale=np.ones(FEATS["frac"].shape[1]), length_scale_bounds=(1e-2, 1e3))
        + WhiteKernel(0.1, (1e-5, 1e1)), normalize_y=True, n_restarts_optimizer=2, random_state=seed))


CANDIDATES = {
    # name: (feature set, constructor)
    "rf_magpie": ("magpie", lambda s: RandomForestRegressor(500, n_jobs=NJ, random_state=s)),  # ~ RF-SCM/Magpie
    "rf_frac": ("frac", lambda s: RandomForestRegressor(500, n_jobs=NJ, random_state=s)),  # ~ RF-Regex Steels
    "et_fracmag": ("frac+magpie", lambda s: ExtraTreesRegressor(1000, n_jobs=NJ, random_state=s)),
    "gbr_frac": ("frac", lambda s: GradientBoostingRegressor(n_estimators=800, learning_rate=0.03, max_depth=4,
                                                             subsample=0.8, max_features=0.8, random_state=s)),
    "gp_frac": ("frac", gp),
    "tabpfn_frac": ("frac", tabpfn),
    "tabpfn_fracmag": ("frac+magpie", tabpfn),
    "tabpfn_magpie": ("magpie", tabpfn),
}
if TASK == "matbench_steels":
    CANDIDATES.pop("tabpfn_magpie")
if TASK != "matbench_steels":  # 4,604 rows: GP and TabPFN+magpie too heavy for this loaded Mac on a first run
    for k in ("gp_frac", "tabpfn_fracmag"):
        CANDIDATES.pop(k)
if os.environ.get("QV_ONLY"):
    CANDIDATES = {k: v for k, v in CANDIDATES.items() if k in os.environ["QV_ONLY"].split(",")}


def fit_predict(name, Xtr, ytr, Xte, seed):
    m = CANDIDATES[name][1](seed)
    m.fit(Xtr, ytr)
    return np.asarray(m.predict(Xte), dtype=float)


# ---------- official folds ----------
NOINNER = bool(os.environ.get("QV_NOINNER"))  # skip inner CV (and qv_select3) for the larger tasks
mbs = {n: MatbenchBenchmark(autoload=False, subset=[TASK]) for n in list(CANDIDATES) + ([] if NOINNER else ["qv_select3"])}
for _mb in mbs.values():
    list(_mb.tasks)[0].load()
summary = {"task": TASK, "candidates": {}, "inner_cv": {}, "timing_s": {}}
for fold in task0.folds:
    tr_in, tr_out = task0.get_train_and_val_data(fold)
    te_in = task0.get_test_data(fold, include_target=False)
    ytr = tr_out.values.astype(float)
    preds, inner = {}, {}
    kf = KFold(5, shuffle=True, random_state=100 + fold)
    for name, (kind, _) in CANDIDATES.items():
        t = time.time()
        Xtr, Xte = X_of(kind, tr_in), X_of(kind, te_in)
        preds[name] = fit_predict(name, Xtr, ytr, Xte, fold)
        oof = np.zeros_like(ytr)
        for a, b in ([] if NOINNER else kf.split(Xtr)):  # inner CV on the TRAINING fold only (used for qv_select3)
            oof[b] = fit_predict(name, Xtr[a], ytr[a], Xtr[b], fold)
        inner[name] = float(np.mean(np.abs(oof - ytr)))
        summary["timing_s"].setdefault(name, 0.0)
        summary["timing_s"][name] += time.time() - t
        log(f"fold {fold} {name}: inner-CV MAE {inner[name]:.4f} ({time.time() - t:.0f} s)")
    top3 = sorted(inner, key=inner.get)[:3]
    if not NOINNER:
        preds["qv_select3"] = np.mean([preds[n] for n in top3], axis=0)
    summary["inner_cv"][fold] = {"mae": inner, "top3": top3}
    for name, p in preds.items():
        list(mbs[name].tasks)[0].record(fold, p)
    log(f"fold {fold} recorded; qv_select3 = mean of {top3}")

for name, mb in mbs.items():
    t = list(mb.tasks)[0]
    sc = t.scores
    summary["candidates"][name] = {"mae_mean": sc["mae"]["mean"], "mae_std": sc["mae"]["std"],
                                   "rmse_mean": sc["rmse"]["mean"], "max_error_max": sc["max_error"]["max"]}
    mb.add_metadata({"quantavane_model": name, "features": CANDIDATES.get(name, ("top-3 by inner CV",))[0]})
    mb.to_file(os.path.join(OUT, f"results_{name}.json.gz"))
    log(f"{name}: MAE {sc['mae']['mean']:.4f} +- {sc['mae']['std']:.4f}  RMSE {sc['rmse']['mean']:.4f}")
json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=1, default=str)
log("done")
